In [ ]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt

In [ ]:
file_path = '/content/drive/MyDrive/dati/dataset.tsv'
try:
    dataset = pd.read_csv(file_path, delimiter="\t")
    # Remove the first column named 'Unnamed: 0' if it exists
    if 'Unnamed: 0' in dataset.columns:
        dataset = dataset.drop('Unnamed: 0', axis=1)
except Exception as e:
    print(f"An error occurred while loading the file: {e}")
dataset.head()

,#Pdb,PDB,Chain,Mutation(s)_PDB,Mutation(s)_cleaned,iMutation_Location(s),Affinity_mut (M),Affinity_mut_parsed,Affinity_wt (M),Affinity_wt_parsed,actual,ddg,SKEMPI version,Uniprot
0,1CSE_E_I,1CSE,I,LI45G,LI38G,COR,5.26E-11,5.260000e-11,1.12E-12,1.120000e-12,-2.281,2.280577,1,"'P00780', 'P01051'"
1,1CSE_E_I,1CSE,I,LI45S,LI38S,COR,8.33E-12,8.330000e-12,1.12E-12,1.120000e-12,-1.189,1.188776,1,"'P00780', 'P01051'"
2,1CSE_E_I,1CSE,I,LI45P,LI38P,COR,1.02E-07,1.020000e-07,1.12E-12,1.120000e-12,-6.765,6.765446,1,"'P00780', 'P01051'"
3,1CSE_E_I,1CSE,I,LI45I,LI38I,COR,1.72E-10,1.720000e-10,1.12E-12,1.120000e-12,-2.982,2.982502,1,"'P00780', 'P01051'"
4,1CSE_E_I,1CSE,I,LI45D,LI38D,COR,1.92E-09,1.920000e-09,1.12E-12,1.120000e-12,-4.412,4.411843,1,"'P00780', 'P01051'"


In [ ]:
pdb = dataset['PDB'].to_numpy()
chain = dataset['Chain'].to_numpy()
chains = list(np.unique(pdb+'_'+chain))

In [ ]:
import requests
import json
import pandas as pd

with open('pdb.txt', 'r') as reader:
  pdbs = [pdb_id.strip() for pdb_id in reader.readline().split(',') if pdb_id.strip()]

PDB = []
CHAIN = []
UNIPROT = []
for id in pdbs:
  url = f"https://www.ebi.ac.uk/pdbe/api/mappings/uniprot/{id}"
  response = requests.get(url)
  response.raise_for_status()
  data = response.json()
  pdb = id.lower()
  chain_to_uniprot = {}
  for uniprot, info in data[pdb]["UniProt"].items():
    for m in info["mappings"]:
      chain_to_uniprot[m["chain_id"]] = uniprot
  for key in chain_to_uniprot:
    PDB.append(id)
    CHAIN.append(key)
    UNIPROT.append(chain_to_uniprot[key])

mapping = pd.DataFrame({'PDB':PDB, 'CHAIN':CHAIN, 'UNIPROT':UNIPROT})
mapping.head()

,PDB,CHAIN,UNIPROT
0,1A22,A,P01241
1,1A22,B,P10912
2,1A4Y,A,P13489
3,1A4Y,D,P13489
4,1A4Y,B,P03950


In [ ]:
mapping.to_csv('PDB_UNIPROT.tsv', sep = '\t')

In [ ]:
mapping.shape

(1030, 3)

In [ ]:
unips = mapping['UNIPROT'].value_counts().to_dict()


In [ ]:
len(unips)

322

In [ ]:
unpdb = mapping.groupby('UNIPROT')['PDB'].apply(list).to_dict()
pdbun = mapping.groupby('PDB')['UNIPROT'].apply(list).to_dict()



In [ ]:
len(unpdb)

322

In [ ]:
len(pdbun)

319

In [ ]:
pdb = mapping['PDB'].to_numpy()
chain = mapping['CHAIN'].to_numpy()
chains = np.unique(pdb+'_'+chain)

In [ ]:
len(chains)

1352